# Facial Expression Analysis with OpenCV + MobileNetV2

This notebook provides a simple baseline for facial expression analysis using:
- **OpenCV** for face detection and webcam capture
- **MobileNetV2** for emotion classification

It includes:
1. Model definition (transfer learning)
2. Optional training pipeline from folders
3. Real-time webcam inference

In [3]:
! pip install opencv-python tensorflow

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached keras-3.14.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.5.4-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cached


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import cv2
import numpy as np
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('OpenCV:', cv2.__version__)

TensorFlow: 2.21.0
OpenCV: 4.13.0


In [ ]:
IMAGE_SIZE: tuple[int, int] = (224, 224)
EMOTION_LABELS: list[str] = [
    'angry',
    'disgust',
    'fear',
    'happy',
    'neutral',
    'sad',
    'surprise',
]

DATA_ROOT: Path = Path('data/fer_like')
TRAIN_DIR: Path = DATA_ROOT / 'train'
VAL_DIR: Path = DATA_ROOT / 'val'
MODEL_DIR: Path = Path('models')
MODEL_WEIGHTS_PATH: Path = MODEL_DIR / 'emotion_mobilenetv2.weights.h5'


def load_face_detector() -> cv2.CascadeClassifier:
    """Load and validate the OpenCV Haar cascade face detector.

    Returns:
        A configured Haar cascade classifier for frontal face detection.

    Raises:
        RuntimeError: If the cascade file cannot be loaded.
    """
    cascade_path: str = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    detector = cv2.CascadeClassifier(cascade_path)
    if detector.empty():
        raise RuntimeError(f'Could not load Haar cascade from: {cascade_path}')
    return detector


def build_emotion_model(num_classes: int) -> tf.keras.Model:
    """Build a simple MobileNetV2 transfer-learning model for emotion classification.

    Args:
        num_classes: Number of output emotion classes.

    Returns:
        A compiled Keras model ready for training or inference.
    """
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='emotion')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='emotion_mobilenetv2')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model


model = build_emotion_model(num_classes=len(EMOTION_LABELS))
model.summary()

In [ ]:
def create_datasets(
    train_dir: Path,
    val_dir: Path,
    image_size: tuple[int, int],
    batch_size: int = 32,
) -> tuple[tf.data.Dataset, tf.data.Dataset]:
    """Create TensorFlow datasets for supervised training.

    Args:
        train_dir: Directory with class subfolders for training samples.
        val_dir: Directory with class subfolders for validation samples.
        image_size: Target spatial size used by the model.
        batch_size: Number of samples per batch.

    Returns:
        A tuple with (train_dataset, validation_dataset).

    Raises:
        FileNotFoundError: If one or both dataset directories do not exist.
    """
    if not train_dir.exists() or not val_dir.exists():
        raise FileNotFoundError(
            f'Dataset folders not found. Expected: {train_dir} and {val_dir}'
        )

    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels='inferred',
        label_mode='int',
        image_size=image_size,
        batch_size=batch_size,
        shuffle=True,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        labels='inferred',
        label_mode='int',
        image_size=image_size,
        batch_size=batch_size,
        shuffle=False,
    )

    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(autotune)
    val_ds = val_ds.prefetch(autotune)
    return train_ds, val_ds


def train_if_data_exists(model: tf.keras.Model, epochs: int = 5) -> None:
    """Train the model only when dataset folders exist, then save weights.

    Args:
        model: The compiled Keras model.
        epochs: Number of training epochs.
    """
    if not TRAIN_DIR.exists() or not VAL_DIR.exists():
        print('Dataset folders not found. Skipping training.')
        print(f'Expected folders: {TRAIN_DIR} and {VAL_DIR}')
        return

    train_ds, val_ds = create_datasets(TRAIN_DIR, VAL_DIR, IMAGE_SIZE, batch_size=32)
    model.fit(train_ds, validation_data=val_ds, epochs=epochs)

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save_weights(MODEL_WEIGHTS_PATH)
    print(f'Saved weights to: {MODEL_WEIGHTS_PATH}')

In [ ]:
# Optional: run training when your dataset is available under data/fer_like/train and data/fer_like/val
train_if_data_exists(model=model, epochs=5)

In [ ]:
def preprocess_face_roi(face_bgr: np.ndarray, image_size: tuple[int, int]) -> np.ndarray:
    """Convert a face ROI into a normalized batch for model inference.

    Args:
        face_bgr: Face region in BGR color space from OpenCV.
        image_size: Target image size expected by the model.

    Returns:
        A NumPy batch array with shape (1, H, W, 3).
    """
    resized = cv2.resize(face_bgr, image_size, interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    batch = np.expand_dims(rgb.astype(np.float32), axis=0)
    return batch


def predict_emotion(
    model: tf.keras.Model,
    face_bgr: np.ndarray,
    labels: Sequence[str],
) -> tuple[str, float]:
    """Predict emotion label and confidence for a single face image.

    Args:
        model: Trained emotion classifier.
        face_bgr: Face image (BGR).
        labels: Ordered list of label names matching model outputs.

    Returns:
        A tuple with predicted label and confidence score in [0, 1].
    """
    batch = preprocess_face_roi(face_bgr=face_bgr, image_size=IMAGE_SIZE)
    probs = model.predict(batch, verbose=0)[0]
    class_idx = int(np.argmax(probs))
    confidence = float(probs[class_idx])
    return labels[class_idx], confidence


def run_webcam_inference(
    model: tf.keras.Model,
    detector: cv2.CascadeClassifier,
    labels: Sequence[str],
    camera_index: int = 0,
    min_confidence: float = 0.35,
) -> None:
    """Run real-time emotion inference from webcam using face detection.

    Args:
        model: Emotion classifier model.
        detector: OpenCV face detector.
        labels: Output labels in model order.
        camera_index: OpenCV camera index.
        min_confidence: Minimum confidence to display predicted label.

    Raises:
        RuntimeError: If webcam cannot be opened.
    """
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError('Could not open webcam. Check camera permissions/device index.')

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = detector.detectMultiScale(
                gray,
                scaleFactor=1.1,
                minNeighbors=5,
                minSize=(60, 60),
            )

            for (x, y, w, h) in faces:
                face_roi = frame[y:y + h, x:x + w]
                label, conf = predict_emotion(model=model, face_bgr=face_roi, labels=labels)
                shown_label = label if conf >= min_confidence else 'uncertain'

                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                cv2.putText(
                    frame,
                    f'{shown_label}: {conf:.2f}',
                    (x, max(0, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA,
                )

            cv2.imshow('OpenCV + MobileNetV2 Emotion Demo (press q to quit)', frame)
            if (cv2.waitKey(1) & 0xFF) == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [ ]:
# Load trained weights if available; otherwise, inference will use ImageNet-initialized head (not meaningful for emotions).
if MODEL_WEIGHTS_PATH.exists():
    model.load_weights(MODEL_WEIGHTS_PATH)
    print(f'Loaded weights from: {MODEL_WEIGHTS_PATH}')
else:
    print('Trained weights not found. Train first for useful emotion predictions.')

face_detector = load_face_detector()
run_webcam_inference(
    model=model,
    detector=face_detector,
    labels=EMOTION_LABELS,
    camera_index=0,
    min_confidence=0.35,
)

## Folder Layout Expected for Training

Use this structure for optional training:

```
data/fer_like/
  train/
    angry/
    disgust/
    fear/
    happy/
    neutral/
    sad/
    surprise/
  val/
    angry/
    disgust/
    fear/
    happy/
    neutral/
    sad/
    surprise/
```